# Coil-field evaluation and fieldline tracing with GVEC

In this notebook we showcase how one can utilize filament coils in GVEC for evaluating the magnetic field and perform fieldline tracing.
In GVEC all coils are discretized via represented as line segments. Therefore, we need to specify the vertices of the line segments in Cartesian coordinates and the corresponding current of the coil (similar to the coils format as used, e.g. by [MAKEGRID](https://princetonuniversity.github.io/STELLOPT/MAKEGRID)). As this might be a common use-case, let us create a `CoilSet`, that is a collection of coils, from such a coils-formatted file. Here, we utilize the [NCSX coils](https://princetonuniversity.github.io/STELLOPT/examples/coils.c09r00) openly available from the [STELLOPT](https://princetonuniversity.github.io/STELLOPT/VMEC%20Free%20Boundary%20Run) examples.

In GVEC, several `Coils` can be combined to a `CoilSet`, so that the total field from all coils in the `CoilSet` can be easily evaluated. As Mentioned above, we have top specify vertices and the coil current to define a coil. Thus, we extract these vertices and currents from the `coils.c09r00` file via pandas:

In [ ]:
import pandas as pd
import numpy as np
from gvec.coils import Coil, CoilSet

# read the coil file
coil_data = pd.read_csv(
    "coils.c09r00", sep=r"\s+", usecols=[0, 1, 2, 3], skiprows=3, names=["x", "y", "z", "I"]
)[:-1]

# find the separation of the coils
idx_coil_end = np.where(coil_data["I"] == 0.0)

# extract the currents
coil_currents = [coil_data["I"][i - 1] for i in idx_coil_end[0]]

# extract the verices
coil_points = np.array(coil_data).astype(float)[:, :-1].T

# gather all coils to combine them into a CoilSet
coils = []
offset = 0
for i, end_idx in enumerate(idx_coil_end[0]):
    # create a single Coil
    coils.append(Coil(coil_points[:, offset : end_idx + 1], coil_currents[i]))
    offset = end_idx + 1

coil_set = CoilSet(coils)

Once we have the `CoilSet`, it can be easily visualized using `plotly`:

In [ ]:
coil_set.plot();

Note that we could also assign names to the coils when creating the `CoilSet` by passing a list of strings via the `coil_names` argument, if no names are passed the coils will just be numbered. Individual coils in the `CoilSet` can be accessed via this name:

In [ ]:
coil_set["coil_0"].plot();

To evaluate the magnetic field at specific points, one has to pass the Cartesian coordinates of the evaluation points to the `eval_B` routine of the `Coil` or `CoilSet` object. Note that this can also be the `pos` quantity from a GVEC evaluation data-set. 

In [ ]:
n_eval_pos = 10
eval_pos = np.zeros((3, n_eval_pos))
eval_pos[0, :] = np.linspace(0.436, 2.435, n_eval_pos)
eval_pos[2, :] = np.linspace(-1, 1, n_eval_pos)

coil_set.eval_B(eval_pos)

In [ ]:
coil_set["coil_0"].eval_B(eval_pos)

Finally, once coils have been translated to GVEC, one can more easily save and load them:

In [ ]:
coil_set.save("NCSX_coils.nc")
del coil_set
coil_set = CoilSet.load("NCSX_coils.nc")
coil_set.plot();

## Fieldline tracing

With a given `CoilSet`, we can also perform fieldline tracing to, e.g., examine the vacuum field. In GVEC, the core routine for this is `trace_fieldlines`: This routine traces fieldlines in Carthesian coordinates utilizing [`scipy.integrate.solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) under the hood. It returns a data-tree that contains each fieldline as a data-set, tracking the fieldline position over time. The essential inputs to `trace_fieldlines` are the starting positions, the coil_set and the time for how long to trace the fieldlines. In addition to the inputs to [`solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html), one can also provide surface normals and surface points to specify planes on which intersections with the fieldlines are tracked, that is for Poincaré plots. In the example below we utilize the NCSX coilset to trace fieldlines.

In [ ]:
from gvec.coils import trace_fieldlines

n_fieldlines = 10  # number of initialized fieldlines
n_planes = 4  # nuber of intersection planes for Poincaré plots

# define starting positions for the fieldlines in R/Z/phi
R_start = np.linspace(1.45, 1.55, n_fieldlines)
Z_start = np.zeros_like(R_start)
phis_start = np.zeros_like(R_start)

# limit the region where the fieldline is traced
trace_limits_R = [0.8, 1.8]
trace_limits_Z = [-0.7, 0.7]


# define a terminating event for the fieldline tracing
def bounding_box_event(t, xyz):
    R = np.sqrt(xyz[0] ** 2 + xyz[1] ** 2)
    if (trace_limits_R[0] <= R <= trace_limits_R[1]) and trace_limits_Z[0] <= xyz[
        2
    ] <= trace_limits_Z[1]:
        return -1
    else:
        return 1


bounding_box_event.terminal = True

# translate R/Z/Phi starts into xyz
starts = np.zeros([3, n_fieldlines])
starts[0, :] = R_start * np.cos(phis_start)
starts[1, :] = R_start * np.sin(phis_start)
starts[2, :] = Z_start

# define surface normals of for of the intersection planes
phis = [0, np.pi / 6, np.pi / 3, np.pi / 2]
surf_normals = [np.array([np.cos(phi), np.sin(phi), 0]) for phi in phis]

dt = trace_fieldlines(
    starts=starts,  # initial conditions for the fiedlines
    coils=coil_set,  # Object for avaluating the B-field
    t=400,  # time to trace fieldlines
    surf_normals=surf_normals,  # intersection planes
    # surf_points = None # note that without surf_points the planes are expected to go trough the origin
    n_jobs=min(n_fieldlines, 5),  # parallelization over fieldlines using joblib
    # solve_ivp specific keywords
    events=[bounding_box_event],
    atol=1e-8,  # tune these tol. for more accurate results
    rtol=1e-5,
)

Intersection positions can be extracted for each fieldline via the `event` entry in the data set, numbered according to the order of the specified surface normals. Below we showcase a possible way to visualize the Poincaré sections using the returned data-tree.

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 2, figsize=(8, 8), tight_layout=True)
for fieldline in dt:
    # select the relevant fieldline
    ds = dt[fieldline]
    try:  # continue if the fieldline never intersected with any plane
        for i, ax in enumerate(axs.flatten()):
            # select intersection positions for the i-th plane
            events = ds[f"event_{i}"]  # corresponds to surface_normal[i]

            # xyz --> RZphi
            R = np.sqrt(events.sel(xyz="x") ** 2 + events.sel(xyz="y") ** 2)
            ax.scatter(
                R[::2],  # select every other point to just get one Poincaré section
                events.sel(xyz="z")[::2],
                marker=".",
                s=3,
            )
    except KeyError:
        continue

# add some plot labels and titles
for ax, phi in zip(axs.flatten(), phis):
    ax.set(xlabel="R / m", ylabel="Z / m", title=f"$\\phi$={phi * np.pi:.2f} $\\pi$")
    ax.axis("equal")

With the data in the returned data-tree, we can also perform some 3D visualizations to inspect the fieldlines and Poincaré sections with respect to the coils. Below is one example of how this could be achieved, again using the `plotly` package.

In [ ]:
from plotly import graph_objects as go

# plot the coils
ax = coil_set.plot(
    show=False,
    line={"color": "black"},
    showlegend=False,
)
ax["layout"]["scene"]["aspectmode"] = "data"

for fieldline in dt:
    # plot the field lines
    ds = dt[fieldline]
    ax.add_trace(
        go.Scatter3d(
            x=ds.pos.sel(xyz="x"),
            y=ds.pos.sel(xyz="y"),
            z=ds.pos.sel(xyz="z"),
            mode="lines",
            opacity=0.05,
            line={"color": "green"},
            showlegend=False,
        )
    )
    # plot the Poincaré sections in 3D
    try:
        for i in range(len(phis)):
            ax.add_trace(
                go.Scatter3d(
                    x=np.array(ds[f"event_{i}"].sel(xyz="x")),
                    y=np.array(ds[f"event_{i}"].sel(xyz="y")),
                    z=np.array(ds[f"event_{i}"].sel(xyz="z")),
                    mode="markers",
                    marker=dict(size=2, color="blue"),
                    showlegend=False,
                )
            )
    except KeyError:
        continue
ax.show();